# Notebook 10 — within-person signal and earlier-recording HR reference

**Question:** do individual HR baseline differences obscure useful HR information? This is one
bounded exploratory development experiment, with no promise of improvement. Read the
[frozen specification](../reports/past_hr_reference_specification.md). Corrected inputs and original
participant folds are preserved; all 32 final-test participants remain sealed.

Two analyses are deliberately separate: retrospective full-record person diagnostics, and prediction
using only an earlier unlabeled wearable interval. No full-record person means or EMA answers enter
predictors. Only aggregates and clearly synthetic examples appear in this notebook.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
from garmin_preprocessing.common import verify_bundle  # noqa: E402
from garmin_past_hr_reference.diagnostics import decomposition  # noqa: E402
from garmin_past_hr_reference.reference import earlier_reference  # noqa: E402
from garmin_hr_activity.features import CLOCK_FLAGS, prepare  # noqa: E402

BUNDLE = ROOT / "private/past_hr_reference/runs/510f1ad96a9e46f3"
REPORT = ROOT / "reports/past_hr_reference/d021427887616868"
manifest = verify_bundle(BUNDLE)
verify_bundle(REPORT)
assert manifest["development_only"] and not manifest["final_test_access"]
print("Verified current development bundle; final-test data are not loaded.")

def aggregate(name):
    return pd.read_json(REPORT / (name + ".json"))


## 1. Where does composite variation occur?

Synthetic illustration: one fictional person reports [3,5], another [9,11]. Their means are 4 and
10; both vary around those means. Total squared deviation splits exactly into within-person 4 and
between-person 36. This is a descriptive decomposition, not a reliability coefficient or causal model.

The actual composite remains nervous + (6−cope) + (6−control), with three valid stored 1–5 codes.
Displayed questionnaire anchors are 0–4. Higher raw cope/control values contribute lower stress
scores after reversal. Items are described separately, not selected as replacement targets.
Morning refers to since waking; Midday/Endday refer to the last prompt.

In [ ]:
synthetic_scores = pd.DataFrame({"n_participant": ["synthetic_A", "synthetic_A", "synthetic_B", "synthetic_B"], "pss_combo": [3., 5., 9., 11.]})
example = decomposition(synthetic_scores, "pss_combo")
assert example["within_ss"] == 4 and example["between_ss"] == 36
print({"synthetic_only": True, "within_SS": example["within_ss"], "between_SS": example["between_ss"], "within_fraction": example["within_ss_fraction"]})
display(aggregate("repeated_counts"))
variation = aggregate("variation")
display(variation[["scope", "measure", "valid_responses", "participants", "within_ss_fraction", "between_ss_fraction", "pooled_within_sd", "person_mean_sd"]])
display(Image(filename=str(REPORT / "outcome_variation.png")))

All 125 development participants have repeated valid scores: median 31, range 17–32. Across
3,689 valid scores, 53.5% of total squared variation is within participants and 46.5% between observed
means. On the existing 60-minute HR cohort, the split is 52.3% / 47.7% (1,634 responses / 118 people;
one singleton). These are response-count-weighted SS fractions, not random-effects ICC estimates.
Observed person means can reflect differences in survey/time sampling as well as stable differences.

In [ ]:
items = aggregate("component_items")
display(items[items.survey_category.eq("All")][["scope", "item", "valid", "missing", "stored_mean", "composite_contribution_mean", "code_1", "code_2", "code_3", "code_4", "code_5"]])
display(aggregate("retrospective_associations"))
display(Image(filename=str(REPORT / "retrospective_associations.png")))

Between-person HR–score associations use one full-record mean pair per participant. Within-person
associations center both HR and score within each person, with participant-cluster bootstrap intervals.
For HR mean, between-person r=.096; centered r=−.032. All displayed association intervals include zero.
These full-record means can use future responses relative to an event: they are **retrospective only**,
are not saved at participant level, and never become predictive features. No causal inference follows.

## 2. A past-only earlier-recording reference — synthetic timeline

For cutoff Tuesday 12:00 UTC: current HR is [Tuesday 11:00,Tuesday 12:00); history is
[Monday 11:00,Tuesday 11:00). A reading exactly Tuesday 11:00 belongs to current, not reference.
Clip readings first; then form observed UTC-minute means and average those means equally.
Empty minutes remain missing. A partial boundary minute uses only inside readings.

The frozen engineering rule requires **30 occupied minutes across three UTC-hour bins**. It is not
a physiological standard or resting-HR definition. For the synthetic values below, ten minute means
at 60, ten at 70 and ten at 80 give reference 70. Current mean 85/latest 90 give deviations +15/+20.
The excluded future/current readings cannot change it.

In [ ]:
config = json.loads((ROOT / "config/past_hr_reference.json").read_text())
hr_rules = json.loads((ROOT / "config/hr_activity.json").read_text())
synthetic_times = [3600*h + 60*m for h in [1, 2, 3] for m in range(10)] + [86400, 90000]
synthetic_values = [60.]*10 + [70.]*10 + [80.]*10 + [220., 240.]
source = pd.DataFrame({"n_epoch_s": synthetic_times, "n_value": synthetic_values, "n_logical_row": np.arange(32), "n_source_sha256": "synthetic_source", "n_source_row": np.arange(32)+1, "q_conflicting_timestamp_value": False, **{flag: False for flag in CLOCK_FLAGS}})
_, ledger = prepare(source, "hr", hr_rules)
reference, quality, _ = earlier_reference(ledger, 90000., config)
assert reference == 70 and quality["accepted_readings"] == 30
print({"synthetic_only": True, "reference_mean": reference, "current_mean_deviation": 85-reference, "current_latest_deviation": 90-reference, "occupied_minutes": quality["occupied_utc_minutes"], "occupied_hours": quality["occupied_utc_hours"]})

The actual current mean remains reading-weighted; reference is observed-minute-weighted. Deviations
compare those summaries, not a pure change in resting physiology. No current/past EMA answer,
future wearable record or whole-person normalization defines the reference. Held-out participants
may supply only their own earlier unlabeled wearable history: prediction **after an observation
period**, with acquisition/receipt timing still provisional.

In [ ]:
print({key: manifest[key] for key in ["accounted_responses", "reference_available", "native_hr_responses", "eligible_responses", "eligible_participants"]})
composition = aggregate("composition")
display(composition[composition.scope.eq("native_60m_hr")])
attrition = aggregate("attrition")
display(attrition[attrition.scope.eq("native_60m_hr") & attrition.kind.eq("exclusive_combination")])
display(Image(filename=str(REPORT / "history_availability.png")))
quality_summary = aggregate("reference_quality")
display(quality_summary[quality_summary.scope.eq("native_60m_hr") & quality_summary.survey_category.eq("All")][["group", "responses", "occupied_utc_minutes_median", "occupied_utc_hours_median", "latest_reading_age_seconds_median", "maximum_internal_gap_seconds_median", "maximum_gap_including_edges_seconds_median"]])

Reference history passes for 3,059/3,706 responses, independently of current HR/outcome eligibility.
The matched cohort retains 1,575/1,634 native HR responses (96.4%), from 116 people. All responses,
including 59 history exclusions and 17 missing scores overall, remain accounted for privately.
Small-cell public score/quality summaries are suppressed; counts remain visible.

Eligible histories have median 572 occupied minutes and 12 UTC-hour bins, but a median largest gap
including edges of **11.46 hours**. Latest history readings just beyond one hour old do not imply a
complete day of measurement. Occupied bins, gaps and ages stay outside the predictor list.

## 3. A–E on identical training and validation responses

A: matching mean/median dummies. B: time/category context. C: context + four raw HR summaries.
D: context + reference mean only. E: context + reference + mean/latest deviations + current SD/slope.

Reuse the exact Notebook 09 Ridge/HGB grids. Five original outer participant folds contain three
participant-grouped inner folds, selecting by mean inner MAE. Each training split fits its own scaler
and encoder. No tuning expansion, row-random early stopping, imputation, quality predictors or answer
history. Ridge/HGB comparisons include different losses and penalties. E−C is a representation/reference
comparison; E−D asks whether current physiology adds anything conditional on reference. Neither is causal.

In [ ]:
metrics = aggregate("metrics")
pooled = metrics[metrics.stage.eq("oof") & metrics.fold.eq("pooled") & metrics.survey_category.eq("All")]
display(pooled[["model", "responses", "participants", "mae", "rmse", "participant_mae", "prediction_sd"]])
display(aggregate("fold_counts"))
display(Image(filename=str(REPORT / "matched_errors.png")))
contrasts = aggregate("paired_intervals")
display(contrasts[contrasts.model.str.contains("E_reference")])
display(Image(filename=str(REPORT / "paired_differences.png")))

## 4. Results and uncertainty

HGB E MAE is **2.0949**, versus C **2.0646**, D **2.0854** and median **2.0673**.
E−C: **+.0303 [−.0253,+.0840]**; E−D: **+.0095 [−.0172,+.0343]**. Ridge E is also worse in point
estimate (2.1218 versus C 2.1158 and D 2.1144). The new reference/deviation representation does not
establish a predictive advantage. Raw-HR HGB's tiny .0027-point advantage over median is uncertain.

Paired bootstrap intervals resample whole participants with their fixed OOF predictions, 2,000 draws.
They omit model-refitting/tuning uncertainty, multiple-comparison correction and external validation.
Nested tuning does not undo repeated development exploration. No winning model or automatic expansion.

In [ ]:
settings = aggregate("selected_settings")
display(settings.groupby(["model", "setting"]).size().rename("folds_selected").reset_index())
display(settings[["fold", "model", "training_mae", "validation_mae", "prediction_sd", "unique_predictions", "warnings"]])
display(aggregate("feature_status"))
print("Report:", REPORT.relative_to(ROOT) / "report.md")
print("Professor summary:", REPORT.relative_to(ROOT) / "professor_summary.md")
print("Provider request (draft only):", REPORT.relative_to(ROOT) / "provider_request_draft.md")
print("Ignored features, quality, accounting and predictions:", BUNDLE.relative_to(ROOT))

## 5. Next justified action

**Stop automatic model expansion.** The engineering history requirement is feasible, but observed
support is discontinuous and reference/deviation features did not improve matched prediction. Seek
provider evidence about acquisition versus receipt, batching, prompt logs, sensor-quality annotations,
IBI anchoring and missed/rejected beats. The request is drafted locally and has not been sent.

Verified numerical results remain distinct from provisional timing/physiology assumptions. Full-record
person diagnostics never become predictors. Earlier personalization documentation was conceptual
background only; no obsolete features/results were reused. Raw files, immutable bundles, assignments
and all 32 sealed final-test participants are preserved. No commit, push or final-test evaluation.